In [1]:
import numpy as np  # arrays, linear algebra, numerical computing
from qutip import Bloch  # visualization of single-qubit states (Bloch sphere)  
import matplotlib.pyplot as plt  # plotting and visualization
from qiskit import QuantumCircuit #to encode a quantum state for a circuit 

In [2]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd() / "src"))

from quantum_error_corrections import (
    I, X, Y, Z, H, P0, P1, I8, X1, X2, X3, Z1, Z2, Z3, E1_rho,  bit_flip_kraus_nqubits, 
    U_N_qubits, U_one_gate, U_two_gates, controlled_gate, rotation_gate,
    rho, evolve, recovery_bit_flip, depolarizing_kraus_nqubits, correct_phase_flip,  
    recovery_phase_flip, initial_state, apply_hadamards, normalize_state, 
    projectors, born_rule_probs, sample_from_probs, single_qubit_channel_n_register, 
    measure_pure_state, measurement_density_matrix, 
    measure_probs_first_n, sample_measurements_input, sample_probs, rotation_channel, 
    bit_flip_kraus, phase_flip_kraus, amplitude_damping_kraus, encode_3_qubit_phase_flip_code, 
    phase_damping_kraus, depolarizing_kraus, pauli_kraus_channel, bit_flip_channel_3qubits, syndrome_measurement_bit_flip,  
    apply_channel, apply_kraus, E1_rho, bloch_visualization, correct_bit_flip, syndrome_measurement_phase_flip, 
    dm, random_pure_state, ket0, ket1, ket_plus, bloch_vector, encode_3_qubit_bit_flip_code, ket_minus, apply_kraus_sparse, 
    buildSparseGateSingle, buildSparseCNOT, dm_sparse, ket0_sparse, bit_flip_kraus_nqubits_sparse, doMeasurement
)

1. The circuit for encoding the three-qubit bit-flip code is shown in Circuit_1 belows. Verify that this circuit indeed implements such an encoding.

In [3]:
qc = QuantumCircuit(3)
qc.cx(0, 1)
qc.barrier()
qc.cx(0, 2) 
print("Circuit_1")
print(qc.draw())

Circuit_1
           ░      
q_0: ──■───░───■──
     ┌─┴─┐ ░   │  
q_1: ┤ X ├─░───┼──
     └───┘ ░ ┌─┴─┐
q_2: ──────░─┤ X ├
           ░ └───┘


In [4]:
psi = np.array([0.5, 0.5])  
encoded_psi = encode_3_qubit_bit_flip_code(psi)

print("Encoded 3-qubit state:\n", encoded_psi)

Encoded 3-qubit state:
 [0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]


2) Argue why the three-qubit bit-flip code does not protect against phase-flip errors

Solutions 

The 3-qubit bit-flip code encodes:
|0>_L = |000>
|1>_L = |111>
So any logical state becomes:
|psi>_L = a|000> + b|111>

A phase-flip error is a Z error on one qubit (example: Z1).
 Z|0> = |0>,   Z|1> = -|1>

Apply Z1 to the encoded state:
 Z1|psi>_L = a|000> - b|111>

This state is still inside the code space span{|000>,|111>}.
 Therefore, the phase-flip error does NOT change the bit-flip syndrome,
so it cannot be detected or corrected.

print("Conclusion: The 3-qubit bit-flip code corrects X (bit-flip) errors, but not Z (phase-flip) errors.")

to show this in measurement it is very good used the problems we encoded from Problem 1

|psi> = a|0> + b|1>
|psi>_L = a|000> + b|111> (Logical qubits)

The encoded state is saved in psi_out and used for belows. Phase-flip gate Z (1 qubit) b

In [5]:
psi = np.array([0.5, 0.5])
encoded = encode_3_qubit_bit_flip_code(psi)
print("Encoded state:\n", encoded)
print()
psi_after_noise = bit_flip_channel_3qubits(encoded, psi)
print("State after noise:\n", psi_after_noise)

error = np.kron(Z, np.kron(I, I))     # Z ⊗ I ⊗ I
errored_state = error @ encoded
print("Errored state:\n", errored_state)
print()
syndrome = syndrome_measurement_bit_flip(errored_state)
print("Syndrome of errored_state (S1, S2):\n", syndrome)
print()
corrected_state = correct_bit_flip(errored_state)
syndrome = syndrome_measurement_bit_flip(corrected_state)
print("Syndrome of corrected_state (S1, S2):\n", syndrome)
print()
print("Corrected state:\n", corrected_state) 
encoded_psi = encode_3_qubit_bit_flip_code(psi)
print()
corrected_state = correct_bit_flip(errored_state)
print("Corrected state:\n", corrected_state)

Encoded state:
 [0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]

State after noise:
 [0.35355339+0.j 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j
 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j]
Errored state:
 [ 0.5+0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j -0.5+0.j]

Syndrome of errored_state (S1, S2):
 (1, 1)

Syndrome of corrected_state (S1, S2):
 (1, 1)

Corrected state:
 [ 0.5+0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j -0.5+0.j]

Corrected state:
 [ 0.5+0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j -0.5+0.j]


In [6]:
psi = np.array([0.5, 0.5])
encoded = encode_3_qubit_bit_flip_code(psi)
print("Encoded state:\n", encoded)
print()
psi_after_noise = bit_flip_channel_3qubits(encoded, psi)
print("State after noise:\n", psi_after_noise)

error = np.kron(I, np.kron(Z, I))     # Z ⊗ I ⊗ I
errored_state = error @ encoded
print("Errored state:\n", errored_state)
print()
syndrome = syndrome_measurement_bit_flip(errored_state)
print("Syndrome of errored_state (S1, S2):\n", syndrome)
print()
corrected_state = correct_bit_flip(errored_state)
syndrome = syndrome_measurement_bit_flip(corrected_state)
print("Syndrome of corrected_state (S1, S2):\n", syndrome)
print()
print("Corrected state:\n", corrected_state) 
encoded_psi = encode_3_qubit_bit_flip_code(psi)
print()
corrected_state = correct_bit_flip(errored_state)
print("Corrected state:\n", corrected_state)

Encoded state:
 [0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]

State after noise:
 [0.35355339+0.j 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j
 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j]
Errored state:
 [ 0.5+0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j -0.5+0.j]

Syndrome of errored_state (S1, S2):
 (1, 1)

Syndrome of corrected_state (S1, S2):
 (1, 1)

Corrected state:
 [ 0.5+0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j -0.5+0.j]

Corrected state:
 [ 0.5+0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j  0. +0.j -0.5+0.j]


In [7]:
psi = np.array([0.5, 0.5])
encoded = encode_3_qubit_bit_flip_code(psi)
print("Encoded state:\n", encoded)
print()
psi_after_noise = bit_flip_channel_3qubits(encoded, psi)
print("State after noise:\n", psi_after_noise)

error = np.kron(Z, np.kron(I, Z))     # Z ⊗ I ⊗ I
errored_state = error @ encoded
print("Errored state:\n", errored_state)
print()
syndrome = syndrome_measurement_bit_flip(errored_state)
print("Syndrome of er rored_state (S1, S2):\n", syndrome)
print()
corrected_state = correct_bit_flip(errored_state)
syndrome = syndrome_measurement_bit_flip(corrected_state)
print("Syndrome of corrected_state (S1, S2):\n", syndrome)
print()
print("Corrected state:\n", corrected_state) 
encoded_psi = encode_3_qubit_bit_flip_code(psi)
print()
corrected_state = correct_bit_flip(errored_state)
print("Corrected state:\n", corrected_state)

Encoded state:
 [0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]

State after noise:
 [0.35355339+0.j 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j
 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j 0.35355339+0.j]
Errored state:
 [0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]

Syndrome of er rored_state (S1, S2):
 (1, 1)

Syndrome of corrected_state (S1, S2):
 (1, 1)

Corrected state:
 [0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]

Corrected state:
 [0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]


Conclusion
Above all, we see the following points: 
The 3-qubit bit-flip code can correct X (bit-flip) errors.
It cannot detect or correct Z (phase-flip) errors. It is impossible to detect them in the three-qubit repetition code because the syndrome measurement gives the same result. This happens because Z only changes the relative phase, and the state remains inside the same code space. 
span{∣000⟩,∣111⟩}